# 不同反应通量

In [19]:
#!/usr/bin/env python
"""
综合代谢通量分析工具
集成辅因子成本分析、代谢物通量分析和反应通量查询功能
"""

import cobra
from cobra.flux_analysis import pfba
import numpy as np
import json
import pandas as pd
from script.ECMpy_function import get_enzyme_constraint_model_with_reaction_set_exclusion


class MetabolicFluxAnalyzer:
    def __init__(self, model_file):
        """初始化分析器"""
        print("🔬 " + "="*70)
        print("        综合代谢通量分析工具")
        print("   Integrated Metabolic Flux Analysis Tool")
        print("="*72)

        self.model = get_enzyme_constraint_model_with_reaction_set_exclusion(model_file)
        self.solution = None
        self.target_flux = 0
        self.target_product_id = None

        print(f"✅ 模型加载成功: {len(self.model.reactions)} 反应, {len(self.model.metabolites)} 代谢物")

    def setup_model(self, target_product, custom_constraints=None, glucose_uptake="EX_cpd00027_e_reverse", max_uptake=10):
        """
        配置模型参数

        Parameters:
        -----------
        target_product : str
            目标产物反应ID
        custom_constraints : dict, optional
            自定义反应约束，格式: {'reaction_id': (lower_bound, upper_bound)}
        glucose_uptake : str
            葡萄糖摄取反应ID
        max_uptake : float
            最大摄取量
        """
        print(f"\n⚙️  模型配置")
        print("-" * 50)

        # 设置目标产物
        self.model.objective = target_product
        self.target_product_id = target_product
        print(f"🎯 目标产物: {target_product}")

        # 设置碳源摄取
        if glucose_uptake in self.model.reactions:
            self.model.reactions.get_by_id(glucose_uptake).bounds = (0, max_uptake)
            print(f"🍯 碳源摄取上限: {max_uptake} mmol/gDCW/h")

        # 应用自定义约束
        if custom_constraints:
            print(f"🔧 应用自定义反应约束:")
            for rxn_id, bounds in custom_constraints.items():
                if rxn_id in self.model.reactions:
                    self.model.reactions.get_by_id(rxn_id).bounds = bounds
                    print(f"   {rxn_id}: [{bounds[0]}, {bounds[1]}]")
                else:
                    print(f"   ⚠️  反应 {rxn_id} 不存在于模型中")

        return self

    def optimize_model(self):
        """执行pFBA优化"""
        print(f"\n🧮 执行pFBA优化")
        print("-" * 50)

        try:
            self.solution = pfba(self.model, fraction_of_optimum=1.0)

            if self.solution.status == 'optimal':
                # 获取目标产物通量
                self.target_flux = self.solution.fluxes[self.target_product_id]

                print(f"✅ 优化成功")
                print(f"🎯 目标产物通量: {self.target_flux:.4f} mmol/gDCW/h")

                # 计算产率
                glucose_uptake = abs(self.solution.fluxes.get("EX_cpd00027_e_reverse", 0))
                if glucose_uptake > 1e-6:
                    yield_val = abs(self.target_flux) / glucose_uptake
                    print(f"📊 理论产率: {yield_val:.4f} mol/mol 葡萄糖")

                return True
            else:
                print(f"❌ 优化失败: {self.solution.status}")
                return False

        except Exception as e:
            print(f"❌ 优化错误: {str(e)}")
            return False

    def analyze_cofactor_costs(self, cofactors=None):
        """分析辅因子成本"""
        if cofactors is None:
            cofactors = {
                'ATP': ['cpd00002_c'],
                'NADH': ['cpd00004_c'],
                'NADPH': ['cpd00006_c']
            }

        print(f"\n💰 辅因子成本分析")
        print("=" * 70)

        results = {}

        for cofactor_name, metabolite_ids in cofactors.items():
            total_production = 0
            total_consumption = 0
            production_rxns = []
            consumption_rxns = []

            for met_id in metabolite_ids:
                try:
                    metabolite = self.model.metabolites.get_by_id(met_id)

                    for reaction in metabolite.reactions:
                        coefficient = reaction.metabolites[metabolite]
                        flux = self.solution.fluxes[reaction.id]
                        contribution = coefficient * flux

                        if abs(contribution) > 1e-6:
                            if contribution > 0:
                                total_production += contribution
                                production_rxns.append({
                                    'id': reaction.id,
                                    'flux': flux,
                                    'contribution': contribution
                                })
                            else:
                                total_consumption += abs(contribution)
                                consumption_rxns.append({
                                    'id': reaction.id,
                                    'flux': flux,
                                    'contribution': abs(contribution)
                                })

                except KeyError:
                    print(f"⚠️  代谢物 {met_id} 未找到")

            # 排序
            production_rxns.sort(key=lambda x: x['contribution'], reverse=True)
            consumption_rxns.sort(key=lambda x: x['contribution'], reverse=True)

            results[cofactor_name] = {
                'production': total_production,
                'consumption': total_consumption,
                'net': total_production - total_consumption,
                'production_rxns': production_rxns,
                'consumption_rxns': consumption_rxns
            }

        # 输出结果
        for cofactor, data in results.items():
            print(f"\n💎 {cofactor}")
            print(f"   生产量: {data['production']:>8.3f} mmol/gDCW/h")
            print(f"   消耗量: {data['consumption']:>8.3f} mmol/gDCW/h")
            print(f"   净通量: {data['net']:>8.3f} {'✅' if abs(data['net']) < 1e-4 else '⚠️'}")

            if abs(self.target_flux) > 1e-6:
                cost = data['consumption'] / abs(self.target_flux)
                print(f"   单位成本: {cost:>6.3f} mol {cofactor}/mol 产物")

            # 显示主要反应
            if data['production_rxns']:
                print(f"   🔺 主要生产反应:")
                for rxn in data['production_rxns'][:3]:
                    print(f"      {rxn['id']:20s} +{rxn['contribution']:7.3f}")

            if data['consumption_rxns']:
                print(f"   🔻 主要消耗反应:")
                for rxn in data['consumption_rxns'][:3]:
                    print(f"      {rxn['id']:20s} -{rxn['contribution']:7.3f}")

        return results

    def format_reaction_equation(self, reaction_id):
        """格式化反应方程式"""
        try:
            reaction = self.model.reactions.get_by_id(reaction_id)

            # 分离反应物和产物
            reactants = []
            products = []

            for met, coeff in reaction.metabolites.items():
                met_name = met.name if met.name else met.id
                coeff_str = f"{abs(coeff):.1f}" if abs(coeff) != 1 else ""

                if coeff < 0:  # 反应物
                    reactants.append(f"{coeff_str} {met_name}".strip())
                else:  # 产物
                    products.append(f"{coeff_str} {met_name}".strip())

            # 构建方程式
            reactant_str = " + ".join(reactants)
            product_str = " + ".join(products)

            # 判断可逆性
            arrow = "⇌" if reaction.reversibility else "→"

            equation = f"{reactant_str} {arrow} {product_str}"
            return equation

        except KeyError:
            return f"反应 {reaction_id} 未找到"

    def analyze_metabolite_reactions(self, metabolite_ids, top_n=5):
        """分析特定代谢物的相关反应"""
        if isinstance(metabolite_ids, str):
            metabolite_ids = [metabolite_ids]

        print(f"\n🔍 代谢物反应分析")
        print("=" * 70)

        for met_id in metabolite_ids:
            try:
                metabolite = self.model.metabolites.get_by_id(met_id)
                print(f"\n🧪 {met_id} ({metabolite.name})")
                print("-" * 50)

                # 收集活跃反应
                active_reactions = []
                for rxn in metabolite.reactions:
                    flux = self.solution.fluxes[rxn.id]
                    if abs(flux) > 1e-6:
                        coeff = rxn.metabolites[metabolite]
                        contribution = coeff * flux

                        active_reactions.append({
                            'id': rxn.id,
                            'flux': flux,
                            'contribution': contribution,
                            'equation': self.format_reaction_equation(rxn.id)
                        })

                # 排序并显示
                active_reactions.sort(key=lambda x: abs(x['contribution']), reverse=True)

                total_prod = sum(r['contribution'] for r in active_reactions if r['contribution'] > 0)
                total_cons = sum(abs(r['contribution']) for r in active_reactions if r['contribution'] < 0)

                print(f"总生产: {total_prod:7.3f} | 总消耗: {total_cons:7.3f} | 净通量: {total_prod-total_cons:7.3f}")
                print(f"活跃反应数: {len(active_reactions)}")

                print(f"\n📋 主要反应 (前{min(top_n, len(active_reactions))}个):")
                for i, rxn in enumerate(active_reactions[:top_n], 1):
                    direction = "🟢" if rxn['contribution'] > 0 else "🔴"
                    print(f"{i:2d}. {direction} {rxn['id']:15s} 通量:{rxn['flux']:8.3f} 贡献:{rxn['contribution']:8.3f}")
                    print(f"     {rxn['equation']}")

            except KeyError:
                print(f"❌ 代谢物 {met_id} 未找到")

    def check_reaction_fluxes(self, reaction_ids):
        """检查特定反应的通量"""
        if isinstance(reaction_ids, str):
            reaction_ids = [reaction_ids]

        print(f"\n⚡ 反应通量查询")
        print("=" * 70)

        for rxn_id in reaction_ids:
            if rxn_id in self.model.reactions:
                flux = self.solution.fluxes[rxn_id]
                status = "⚪" if abs(flux) > 1e-6 else "🟢"

                print(f"{status} {rxn_id:25s} {flux:>10.4f} mmol/gDCW/h")

                if abs(flux) > 1e-6:
                    equation = self.format_reaction_equation(rxn_id)
                    print(f"    {equation}")

            else:
                print(f"❌ {rxn_id:25s} 反应不存在")

    def generate_summary_report(self):
        """生成总结报告"""
        print(f"\n📊 分析总结报告")
        print("=" * 70)

        glucose_flux = abs(self.solution.fluxes.get("EX_cpd00027_e_reverse", 0))

        print(f"目标产物反应: {self.target_product_id}")
        print(f"目标产物通量: {self.target_flux:.4f} mmol/gDCW/h")
        print(f"葡萄糖消耗量: {glucose_flux:.4f} mmol/gDCW/h")

        if glucose_flux > 1e-6:
            yield_val = abs(self.target_flux) / glucose_flux
            print(f"碳源转化率: {yield_val:.4f} mol/mol")

        print(f"解状态: {self.solution.status}")
        print(f"目标函数值: {self.solution.objective_value:.4f}")

        # 活跃反应统计
        active_count = sum(1 for flux in self.solution.fluxes if abs(flux) > 1e-6)
        print(f"活跃反应数: {active_count}/{len(self.model.reactions)}")

    def export_flux_to_csv(self, output_file="flux_results.csv", threshold=1e-6, only_active=False):
        """
        将通量结果导出为CSV文件

        Parameters:
        -----------
        output_file : str
            输出文件路径
        threshold : float
            判断活跃的阈值
        only_active : bool
            是否只导出活跃反应（默认False，导出所有反应）

        Returns:
        --------
        str : 输出文件路径
        """
        if self.solution is None:
            print("❌ 请先运行 optimize_model() 进行优化")
            return None

        print(f"\n📁 导出通量结果到CSV文件...")

        # 收集数据
        data = []
        for reaction in self.model.reactions:
            flux = self.solution.fluxes[reaction.id]
            abs_flux = abs(flux)
            is_active = abs_flux > threshold

            # 如果只导出活跃反应，跳过非活跃的
            if only_active and not is_active:
                continue

            data.append({
                'Reaction_ID': reaction.id,
                'Reaction_Name': reaction.name,
                'Flux': flux,
                'Abs_Flux': abs_flux,
                'Is_Active': is_active,
                'Lower_Bound': reaction.lower_bound,
                'Upper_Bound': reaction.upper_bound,
                'Subsystem': reaction.subsystem if reaction.subsystem else '',
                'Equation': reaction.reaction,
                'Gene_Reaction_Rule': reaction.gene_reaction_rule if reaction.gene_reaction_rule else ''
            })

        # 创建DataFrame
        df = pd.DataFrame(data)

        # 按绝对通量排序（降序）
        df = df.sort_values('Abs_Flux', ascending=False)

        # 导出CSV
        df.to_csv(output_file, index=False, encoding='utf-8-sig')

        # 统计信息
        active_count = df['Is_Active'].sum()
        total_model_reactions = len(self.model.reactions)

        print(f"✅ 已导出到: {output_file}")
        print(f"   模型总反应数: {total_model_reactions}")
        print(f"   导出反应数: {len(df)}")
        print(f"   活跃反应数: {active_count}/{total_model_reactions}")

        return output_file

    def export_flux_summary_to_csv(self, output_prefix="flux_analysis", threshold=1e-6):
        """
        导出完整的通量分析结果，包括多个CSV文件

        Parameters:
        output_prefix : str
            输出文件前缀
        threshold : float
            判断活跃的阈值

        Returns:
        --------
        dict : 包含所有输出文件路径的字典
        """
        if self.solution is None:
            print("❌ 请先运行 optimize_model() 进行优化")
            return None

        output_files = {}

        # 1. 导出所有反应通量
        all_flux_file = f"{output_prefix}_all_fluxes.csv"
        self.export_flux_to_csv(all_flux_file, threshold, only_active=False)
        output_files['all_fluxes'] = all_flux_file

        # 2. 导出活跃反应通量
        active_flux_file = f"{output_prefix}_active_fluxes.csv"
        self.export_flux_to_csv(active_flux_file, threshold, only_active=True)
        output_files['active_fluxes'] = active_flux_file

        # 3. 导出统计摘要
        summary_file = f"{output_prefix}_summary.csv"

        # 统计活跃反应
        active_count = sum(1 for flux in self.solution.fluxes if abs(flux) > threshold)
        total_count = len(self.model.reactions)
        active_fluxes = [abs(f) for f in self.solution.fluxes if abs(f) > threshold]

        # 获取葡萄糖摄取量
        glucose_flux = abs(self.solution.fluxes.get("EX_cpd00027_e_reverse", 0))

        summary_data = {
            'Metric': [
                'Total_Reactions',
                'Active_Reactions',
                'Inactive_Reactions',
                'Active_Ratio_%',
                'Objective_Value',
                'Solution_Status',
                'Target_Product',
                'Target_Flux',
                'Glucose_Uptake',
                'Yield_mol_mol',
                'Max_Flux',
                'Min_Active_Flux',
                'Mean_Active_Flux',
                'Analysis_Threshold'
            ],
            'Value': [
                total_count,
                active_count,
                total_count - active_count,
                f"{active_count/total_count*100:.2f}",
                f"{self.solution.objective_value:.6f}",
                self.solution.status,
                self.target_product_id,
                f"{self.target_flux:.6f}",
                f"{glucose_flux:.6f}",
                f"{abs(self.target_flux)/glucose_flux:.4f}" if glucose_flux > 1e-6 else "N/A",
                f"{max(active_fluxes):.6f}" if active_fluxes else "N/A",
                f"{min(active_fluxes):.6f}" if active_fluxes else "N/A",
                f"{sum(active_fluxes)/len(active_fluxes):.6f}" if active_fluxes else "N/A",
                f"{threshold}"
            ]
        }

        summary_df = pd.DataFrame(summary_data)
        summary_df.to_csv(summary_file, index=False, encoding='utf-8-sig')
        output_files['summary'] = summary_file

        print(f"\n📊 统计摘要已导出到: {summary_file}")
        print(f"\n📂 已生成以下文件:")
        for key, filepath in output_files.items():
            print(f"   - {filepath}")

        return output_files


def main():
    """主程序示例"""
    # ========== 配置参数 ==========
    MODEL_FILE = "model_Calibration/DLKcat_Calibration/ecCZ870_DLKcat_Calibration_modify.json"
    TARGET_PRODUCT = "EX_cpd00054_e"
    MODEL_FILE = "ecCGL1111.json"
    TARGET_PRODUCT  = "CG_biomass_cgl_ATCC13032"
    # 自定义反应约束 - 可以设置任何反应的上下限
    CUSTOM_CONSTRAINTS = {
        'EX_cpd00027_e_reverse': (0,10),
        'EX_glc_e_reverse': (0,1000),
        'ATPS4rpp': (0,0),

    }
    # 要分析的代谢物
    TARGET_METABOLITES = ['cpd00061_c','cpd00020_c','cpd00024_c','nadh_c','lac__D_c','fum_c','succ_c','mal__L_c','mql8_c','q8h2_c'
                          ]

    # 要检查的反应
    KEY_REACTIONS= [
        'EX_cpd00027_e_reverse','rxn00006_c','rxn05226_c','rxn01192_c','rxn00558_c','rxn00545_c','rxn00786_c','rxn00747_c_reverse','rxn00781_c','rxn01100_c','rxn01106_c','rxn00459_c','rxn00148_c_num1','rxn00251_c','rxn00250_c','rxn00154_c','rxn00256_c','rxn00974_c','rxn01388_c','rxn00198_c','rxn08094_c','rxn00306_c','rxn00288_c','rxn00161_c','rxn00799_c','rxn00799_c','rxn08901_c','rxn00604_c','rxn15228_c_reverse','rxn00785_c_reverse','rxn01333_c','rxn00184a_c','EX_cpd00011_e','rxn01332_c_num1','rxn00250_c','EX_cpd00039_e','Bio_cgATCC14067_c','rxn00785_c_reverse','rxn00011_c','rxn00336_c','rxn03031_c_num2','EX_cpd00064_e','rxn00161_c','rxnmetB2_c','EX_cpd00119_e','EX_cpd00029_e','rxn00161_c','rxn05289_c_num2','rxn05289_c_num1','rxn00006_c','rxn00161_c',
        TARGET_PRODUCT,'EX_cpd00029_e','rxn00225_c','rxn00799_c_reverse','rxn00902_c','rxn00133_c','rxnmqfadh2_c','rxn00288_c','rxn00173_c','cpd00106_c',

    ]

    # ========== 执行分析 ==========
    try:
        # 初始化分析器
        analyzer = MetabolicFluxAnalyzer(MODEL_FILE)

        # 配置模型
        analyzer.setup_model(
            target_product=TARGET_PRODUCT,
            custom_constraints=CUSTOM_CONSTRAINTS,
            max_uptake=10
        )

        # 优化求解
        if not analyzer.optimize_model():
            print("❌ 优化失败，程序终止")
            return

        # 辅因子成本分析
        analyzer.analyze_cofactor_costs()

        # 代谢物反应分析
        analyzer.analyze_metabolite_reactions(TARGET_METABOLITES,top_n=30)

        # 反应通量查询
        # analyzer.check_reaction_fluxes(KEY_REACTIONS)

        # 生成总结报告
        analyzer.generate_summary_report()

        # analyzer.export_flux_to_csv("active_flux_results.csv", only_active=True)

        print(f"\n✨ 分析完成！")
        print("="*72)

    except Exception as e:
        print(f"❌ 程序执行错误: {str(e)}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

🔬 ======================================================================
        综合代谢通量分析工具
   Integrated Metabolic Flux Analysis Tool
识别反应集合...
总共识别出 0 个需要约束的反应集合
添加反应集合互斥约束...
成功添加了 0 个反应集合互斥约束
✅ 模型加载成功: 1850 反应, 949 代谢物

⚙️  模型配置
--------------------------------------------------
🎯 目标产物: CG_biomass_cgl_ATCC13032
🔧 应用自定义反应约束:
   ⚠️  反应 EX_cpd00027_e_reverse 不存在于模型中
   EX_glc_e_reverse: [0, 1000]
   ATPS4rpp: [0, 0]

🧮 执行pFBA优化
--------------------------------------------------
✅ 优化成功
🎯 目标产物通量: 0.3773 mmol/gDCW/h

💰 辅因子成本分析
⚠️  代谢物 cpd00002_c 未找到
⚠️  代谢物 cpd00004_c 未找到
⚠️  代谢物 cpd00006_c 未找到

💎 ATP
   生产量:    0.000 mmol/gDCW/h
   消耗量:    0.000 mmol/gDCW/h
   净通量:    0.000 ✅
   单位成本:  0.000 mol ATP/mol 产物

💎 NADH
   生产量:    0.000 mmol/gDCW/h
   消耗量:    0.000 mmol/gDCW/h
   净通量:    0.000 ✅
   单位成本:  0.000 mol NADH/mol 产物

💎 NADPH
   生产量:    0.000 mmol/gDCW/h
   消耗量:    0.000 mmol/gDCW/h
   净通量:    0.000 ✅
   单位成本:  0.000 mol NADPH/mol 产物

🔍 代谢物反应分析
❌ 代谢物 cpd00061_c 未找到
❌ 代谢物 cpd00020_c 

## 13032

In [1]:
#!/usr/bin/env python
"""
kcat_MW参数敏感性分析 - 最小化酶池版本（简化版）
针对CGL1代谢模型
使用两阶段优化：固定目标后最小化 Σ(1/kcat_MW × v)
"""

import cobra
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import tempfile
import os
import sys

sys.path.insert(0, '.')

N_JOBS = -1

COLORS = {
    'teal': '#009688',
    'coral': '#E57373',
    'amber': '#FFB74D',
    'purple': '#9575CD',
    'gray': '#757575',
}

# 配置组1：图A中的反应
PLOT_CONFIG_GROUP1 = [
    ('CG_biomass_cgl_ATCC13032', 'Biomass', COLORS['teal'], 'o'),
    ('EX_lac_D_e', 'Lactate', COLORS['coral'], 's'),
    ('EX_o2_e_reverse', r'O$_2$ uptake', COLORS['amber'], '^'),
    ('EX_glc_e_reverse', 'Glucose uptake', COLORS['purple'], 'D'),
]

# 配置组2：图C中的反应（移除ATPS4rpp）
PLOT_CONFIG_GROUP2 = [
    ('CYTBDpp', 'bd-type oxidase', COLORS['coral'], 's'),
    ('CYTBO3_4pp', 'bccaa$_3$ oxidase', COLORS['amber'], '^'),
]

# ATP反应（用于计算比例）
ATP_REACTIONS = ['PGK', 'PYK_num1', 'ATPS4rpp']


def set_academic_style():
    plt.rcParams.update({
        'font.family': 'Arial',
        'font.size': 11,
        'axes.linewidth': 1.0,
        'axes.labelsize': 12,
        'lines.linewidth': 2.0,
        'lines.markersize': 8,
        'xtick.major.width': 1.0,
        'ytick.major.width': 1.0,
        'xtick.labelsize': 10,
        'ytick.labelsize': 10,
        'legend.fontsize': 10,
        'legend.frameon': False,
        'figure.facecolor': 'white',
        'axes.facecolor': 'white',
        'savefig.facecolor': 'white',
    })


def setup_clean_axes(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('black')
    ax.spines['bottom'].set_color('black')
    ax.spines['left'].set_linewidth(1.0)
    ax.spines['bottom'].set_linewidth(1.0)
    ax.grid(False)
    ax.tick_params(axis='both', direction='out', colors='black')


def calculate_atp_proportion(flux_emp1, flux_emp2, flux_synthase):
    atp_emp = flux_emp1 + flux_emp2
    atp_synthase = flux_synthase
    total_atp = atp_emp + atp_synthase
    if total_atp == 0:
        return None
    emp_percentage = (atp_emp / total_atp) * 100
    synthase_percentage = (atp_synthase / total_atp) * 100
    return emp_percentage, synthase_percentage, total_atp


def optimize_minimize_enzyme_pool(model, objective_rxn, fraction_of_optimum=1.0):
    """
    两阶段优化：最小化酶池用量

    目标函数：min Σ(1/kcat_MW × v) + ε × Σv (对无kcat_MW的反应)

    阶段1: 普通FBA → max(objective)
    阶段2: 固定objective → min(酶池用量)
    """

    # 阶段1: 普通FBA找最大目标值
    model.objective = objective_rxn
    model.objective_direction = 'max'

    max_objective = model.slim_optimize()

    if max_objective is None or max_objective <= 1e-9:
        return model.optimize()

    # 阶段2: 固定目标值，最小化酶池
    target_flux = max_objective * fraction_of_optimum
    tolerance = abs(target_flux) * 0.001

    obj_rxn = model.reactions.get_by_id(objective_rxn)
    original_bounds = obj_rxn.bounds
    obj_rxn.bounds = (target_flux - tolerance, target_flux + tolerance)

    # 构建目标函数：对所有反应都加惩罚
    enzyme_cost_objective = {}
    default_penalty = 1e-9  # 对无kcat_MW反应的小惩罚

    for rxn in model.reactions:
        if hasattr(rxn, 'kcat_MW') and rxn.kcat_MW:
            try:
                kcat_mw = float(rxn.kcat_MW)
                if kcat_mw > 0:
                    enzyme_cost_objective[rxn] = -1.0 / kcat_mw
                else:
                    enzyme_cost_objective[rxn] = -default_penalty
            except (ValueError, TypeError):
                enzyme_cost_objective[rxn] = -default_penalty
        else:
            enzyme_cost_objective[rxn] = -default_penalty

    model.objective = enzyme_cost_objective
    model.objective_direction = 'max'  # max(-cost) = min(cost)

    solution = model.optimize()

    obj_rxn.bounds = original_bounds

    return solution


def compute_single_kcat(kcat_mw, model_dict, target_rxn, glucose_bounds):
    """
    使用最小化酶池策略计算单个kcat值
    """
    import sys
    sys.path.insert(0, '.')

    from script.ECMpy_function import get_enzyme_constraint_model_with_reaction_set_exclusion
    import json
    import tempfile
    import os
    import numpy as np

    model_dict_copy = json.loads(json.dumps(model_dict))
    for rxn in model_dict_copy['reactions']:
        if rxn['id'] == target_rxn:
            rxn['kcat_MW'] = kcat_mw
            break

    tmp_path = None
    try:
        with tempfile.NamedTemporaryFile(suffix='.json', mode='w', delete=False) as tmp:
            json.dump(model_dict_copy, tmp, indent=2)
            tmp_path = tmp.name

        model = get_enzyme_constraint_model_with_reaction_set_exclusion(tmp_path)

        # CGL1模型的自定义约束
        custom_constraints = {
            'EX_glc_e_reverse': glucose_bounds,
            'EX_lac_D_e': (0,5),
            'EX_lac_L_e': (0,0),
        }

        for rxn_id, bounds in custom_constraints.items():
            if rxn_id in model.reactions:
                model.reactions.get_by_id(rxn_id).bounds = bounds

        # 使用最小化酶池策略
        solution = optimize_minimize_enzyme_pool(model, 'CG_biomass_cgl_ATCC13032', 1.0)

        if solution is not None and solution.status == 'optimal':
            reactions_to_monitor = [
                'CG_biomass_cgl_ATCC13032', 'CYTBO3_4pp', 'ATPS4rpp', 'CYTBDpp',
                'EX_lac_D_e', 'EX_o2_e_reverse', 'EX_glc_e_reverse'
            ]

            row = {'kcat_MW': kcat_mw}

            for rxn_id in reactions_to_monitor + ATP_REACTIONS:
                row[rxn_id] = solution.fluxes.get(rxn_id, np.nan)

            flux_emp1 = row.get('PGK', 0)
            flux_emp2 = row.get('PYK_num1', 0)
            flux_synthase = row.get('ATPS4rpp', 0)

            atp_info = calculate_atp_proportion(flux_emp1, flux_emp2, flux_synthase)

            if atp_info is not None:
                row['ATP_EMP_percentage'], row['ATP_Synthase_percentage'], row['Total_ATP'] = atp_info
            else:
                row['ATP_EMP_percentage'] = row['ATP_Synthase_percentage'] = np.nan
                row['Total_ATP'] = 0

            return row
        return None

    except Exception as e:
        print(f"Error computing kcat={kcat_mw}: {e}")
        return None
    finally:
        if tmp_path and os.path.exists(tmp_path):
            try:
                os.remove(tmp_path)
            except:
                pass


def find_inactivation_boundary(df, atp_synthase_rxn='ATPS4rpp'):
    """找到ATP合成酶反应通量为0的边界点"""
    if atp_synthase_rxn not in df.columns:
        return None

    threshold = 1e-6
    zero_flux_mask = df[atp_synthase_rxn].abs() < threshold

    if not zero_flux_mask.any():
        return None

    zero_indices = df[zero_flux_mask].index
    if len(zero_indices) > 0:
        return df.loc[zero_indices[0], 'kcat_MW']
    return None


def run_sensitivity_analysis(model_file, target_rxn='ATPS4rpp',
                             kcat_range=(2000, 0.1), n_points=30,
                             glucose_bounds=(0, 1000), n_jobs=None):
    """使用最小化酶池策略并行运行敏感性分析"""
    from joblib import Parallel, delayed

    if n_jobs is None:
        n_jobs = N_JOBS

    print("=" * 80)
    print(f"kcat_MW 敏感性分析 (最小化酶池策略)")
    print(f"优化目标: min Σ(1/kcat_MW × v)")
    print(f"并行任务数: {n_jobs if n_jobs > 0 else '全部CPU'}")
    print(f"修改反应: {target_rxn}")
    print(f"kcat_MW 范围: {kcat_range[0]} → {kcat_range[1]}")
    print(f"葡萄糖约束: {glucose_bounds}")
    print("=" * 80)

    with open(model_file, 'r') as f:
        model_dict = json.load(f)

    kcat_values = np.linspace(kcat_range[0], kcat_range[1], n_points)

    results = Parallel(n_jobs=n_jobs, verbose=0, backend='loky')(
        delayed(compute_single_kcat)(kcat_mw, model_dict, target_rxn, glucose_bounds)
        for kcat_mw in kcat_values
    )

    valid_results = [r for r in results if r is not None]

    df = pd.DataFrame(valid_results)
    if not df.empty:
        df = df.sort_values('kcat_MW', ascending=False).reset_index(drop=True)

    print(f"\n完成！有效结果: {len(valid_results)}/{n_points}")
    return df


def plot_combined_figure(df, save_path='figure_CGL1_minpool.png'):
    set_academic_style()
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

    inactivation_boundary = find_inactivation_boundary(df, 'ATPS4rpp') or 350
    color_reduced = '#E3F2FD'
    color_inactive = '#FFEBEE'

    x_min, x_max = df['kcat_MW'].min(), df['kcat_MW'].max()

    # 子图A
    ax1 = axes[0]
    ax1.axvspan(x_max, inactivation_boundary, alpha=0.3, color=color_reduced, zorder=0)
    ax1.axvspan(inactivation_boundary, x_min, alpha=0.3, color=color_inactive, zorder=0)

    for rxn_id, label, color, marker in PLOT_CONFIG_GROUP1:
        if rxn_id in df.columns:
            ax1.plot(df['kcat_MW'], df[rxn_id], color=color, marker=marker,
                    label=label, linewidth=2.0, markersize=6, markeredgewidth=0, alpha=0.9, zorder=2)

    setup_clean_axes(ax1)
    ax1.set_xlabel(r'$k_{cat}$_MW', fontsize=11)
    ax1.set_ylabel(r'Reaction flux (mmol gDCW$^{-1}$ h$^{-1}$)', fontsize=11)
    ax1.invert_xaxis()
    y_max = df[['CG_biomass_cgl_ATCC13032', 'EX_lac_D_e', 'EX_o2_e_reverse', 'EX_glc_e_reverse']].max().max()
    ax1.set_ylim(bottom=-0.5, top=y_max * 1.25)
    ax1.legend(loc='upper left', frameon=False, fontsize=8, ncol=2)
    ax1.set_title('A', loc='left', fontsize=14, fontweight='bold')

    y1_bottom, y1_top = ax1.get_ylim()
    text_y1 = y1_bottom + (y1_top - y1_bottom) * 0.08
    ax1.text((x_max + inactivation_boundary) / 2, text_y1, 'Reduced activity',
             ha='center', va='center', fontsize=8, color='#1565C0', style='italic')
    ax1.text((inactivation_boundary + x_min) / 2, text_y1, 'Inactivation',
             ha='center', va='center', fontsize=8, color='#C62828', style='italic')

    # 子图B
    ax2 = axes[1]
    ax2.axvspan(x_max, inactivation_boundary, alpha=0.3, color=color_reduced, zorder=0)
    ax2.axvspan(inactivation_boundary, x_min, alpha=0.3, color=color_inactive, zorder=0)

    if not df.empty and 'ATP_EMP_percentage' in df.columns:
        valid_df = df[df['ATP_EMP_percentage'].notna()].copy()
        if not valid_df.empty:
            ax2.plot(valid_df['kcat_MW'], valid_df['ATP_EMP_percentage'],
                    color=COLORS['coral'], marker='o', linewidth=2.0, markersize=6,
                    markeredgewidth=0, label='Substrate-level Phosphorylation', alpha=0.9, zorder=2)
            ax2.plot(valid_df['kcat_MW'], valid_df['ATP_Synthase_percentage'],
                    color=COLORS['teal'], marker='s', linewidth=2.0, markersize=6,
                    markeredgewidth=0, label='Oxidative Phosphorylation', alpha=0.9, zorder=2)
            ax2.axhline(y=50, color=COLORS['gray'], linestyle='--', linewidth=1.0, alpha=0.5, zorder=1)

    setup_clean_axes(ax2)
    ax2.set_xlabel(r'$k_{cat}$_MW', fontsize=11)
    ax2.set_ylabel('ATP synthesis contribution (%)', fontsize=11)
    ax2.invert_xaxis()
    ax2.set_ylim(-2, 102)
    ax2.set_yticks(np.arange(0, 101, 20))
    ax2.legend(loc='upper left', frameon=False, fontsize=8)
    ax2.set_title('B', loc='left', fontsize=14, fontweight='bold')

    y2_bottom, y2_top = ax2.get_ylim()
    text_y2 = y2_bottom + (y2_top - y2_bottom) * 0.08
    ax2.text((x_max + inactivation_boundary) / 2, text_y2, 'Reduced activity',
             ha='center', va='center', fontsize=8, color='#1565C0', style='italic')
    ax2.text((inactivation_boundary + x_min) / 2, text_y2, 'Inactivation',
             ha='center', va='center', fontsize=8, color='#C62828', style='italic')

    # 子图C（移除了ATPS4rpp）
    ax3 = axes[2]
    ax3.axvspan(x_max, inactivation_boundary, alpha=0.3, color=color_reduced, zorder=0)
    ax3.axvspan(inactivation_boundary, x_min, alpha=0.3, color=color_inactive, zorder=0)

    for rxn_id, label, color, marker in PLOT_CONFIG_GROUP2:
        if rxn_id in df.columns:
            ax3.plot(df['kcat_MW'], df[rxn_id], color=color, marker=marker,
                    label=label, linewidth=2.0, markersize=6, markeredgewidth=0, alpha=0.9, zorder=2)

    setup_clean_axes(ax3)
    ax3.set_xlabel(r'$k_{cat}$_MW', fontsize=11)
    ax3.set_ylabel(r'Reaction flux (mmol gDCW$^{-1}$ h$^{-1}$)', fontsize=11)
    ax3.invert_xaxis()
    y3_max = df[['CYTBDpp', 'CYTBO3_4pp']].max().max()
    ax3.set_ylim(bottom=-0.5, top=y3_max * 1.25)
    ax3.legend(loc='upper left', frameon=False, fontsize=8)
    ax3.set_title('C', loc='left', fontsize=14, fontweight='bold')

    y3_bottom, y3_top = ax3.get_ylim()
    text_y3 = y3_bottom + (y3_top - y3_bottom) * 0.08
    ax3.text((x_max + inactivation_boundary) / 2, text_y3, 'Reduced activity',
             ha='center', va='center', fontsize=8, color='#1565C0', style='italic')
    ax3.text((inactivation_boundary + x_min) / 2, text_y3, 'Inactivation',
             ha='center', va='center', fontsize=8, color='#C62828', style='italic')

    plt.tight_layout()
    plt.savefig(save_path, dpi=600, bbox_inches='tight')
    plt.close()
    print(f"图像已保存: {save_path}")


def main():
    MODEL_FILE = "ecCGL1.json"

    if not os.path.exists(MODEL_FILE):
        print(f"模型文件不存在: {MODEL_FILE}")
        return

    print("\n运行敏感性分析（葡萄糖摄入: 0-1000）")
    df = run_sensitivity_analysis(
        model_file=MODEL_FILE,
        target_rxn='ATPS4rpp',
        kcat_range=(800, 10),
        n_points=20,
        glucose_bounds=(0, 1000),
        n_jobs=N_JOBS
    )

    if df.empty:
        print("未获得有效结果")
        return

    plot_combined_figure(df)

    print("\n" + "=" * 100)
    print("分析完成！")
    print("生成的文件:")
    print("  - figure_CGL1_minpool.png (组合图)")
    print("  - kcat_sensitivity_minpool.csv (数据文件)")
    print("=" * 100)

if __name__ == "__main__":
    main()


运行敏感性分析（葡萄糖摄入: 0-1000）
kcat_MW 敏感性分析 (最小化酶池策略)
优化目标: min Σ(1/kcat_MW × v)
并行任务数: 全部CPU
修改反应: ATPS4rpp
kcat_MW 范围: 800 → 10
葡萄糖约束: (0, 1000)

完成！有效结果: 20/20
图像已保存: figure_CGL1_minpool.png

分析完成！
生成的文件:
  - figure_CGL1_minpool.png (组合图)
  - kcat_sensitivity_minpool.csv (数据文件)


## 14067

In [28]:
#!/usr/bin/env python
"""
kcat_MW参数敏感性分析 - 最小化酶池版本（简化版）
修复：对没有kcat_MW的反应也加小惩罚，避免极端通量
"""

import cobra
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import tempfile
import os
import sys

sys.path.insert(0, '.')

N_JOBS = -1

COLORS = {
    'teal': '#009688',
    'coral': '#E57373',
    'amber': '#FFB74D',
    'purple': '#9575CD',
    'gray': '#757575',
}

PLOT_CONFIG_GROUP1 = [
    ('EX_biomass_c', 'Biomass', COLORS['teal'], 'o'),
    ('EX_cpd00221_e', 'Lactate', COLORS['coral'], 's'),
    ('EX_cpd00007_e_reverse', r'O$_2$ uptake', COLORS['amber'], '^'),
    ('EX_cpd00027_e_reverse', 'Glucose uptake', COLORS['purple'], 'D'),
]

PLOT_CONFIG_GROUP2 = [
    ('rxn08287_c', 'bd-type oxidase', COLORS['coral'], 's'),
    ('rxnbccaa3_c', 'bccaa$_3$ oxidase', COLORS['amber'], '^'),
    ('rxn08971_c', 'rxn08971_c', COLORS['teal'], 'o'),
]


def set_academic_style():
    plt.rcParams.update({
        'font.family': 'Arial',
        'font.size': 11,
        'axes.linewidth': 1.0,
        'axes.labelsize': 12,
        'lines.linewidth': 2.0,
        'lines.markersize': 8,
        'xtick.major.width': 1.0,
        'ytick.major.width': 1.0,
        'xtick.labelsize': 10,
        'ytick.labelsize': 10,
        'legend.fontsize': 10,
        'legend.frameon': False,
        'figure.facecolor': 'white',
        'axes.facecolor': 'white',
        'savefig.facecolor': 'white',
    })


def setup_clean_axes(ax):
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_color('black')
    ax.spines['bottom'].set_color('black')
    ax.spines['left'].set_linewidth(1.0)
    ax.spines['bottom'].set_linewidth(1.0)
    ax.grid(False)
    ax.tick_params(axis='both', direction='out', colors='black')


def calculate_atp_proportion(flux_emp1, flux_emp2, flux_synthase):
    atp_emp = flux_emp1 + flux_emp2
    atp_synthase = flux_synthase
    total_atp = atp_emp + atp_synthase
    if total_atp == 0:
        return None
    emp_percentage = (atp_emp / total_atp) * 100
    synthase_percentage = (atp_synthase / total_atp) * 100
    return emp_percentage, synthase_percentage, total_atp


def optimize_minimize_enzyme_pool(model, objective_rxn='EX_biomass_c', fraction_of_optimum=1.0):
    """
    两阶段优化：最小化酶池用量（修复版）

    修复：对没有kcat_MW的反应也加小惩罚系数，避免出现极端通量

    目标函数：
    min Σ(1/kcat_MW × v) + ε × Σv  (对无kcat_MW的反应)

    其中 ε = 1e-9，足够小不影响酶成本优化，但能避免极端解
    """

    # ===== 阶段1: 普通FBA找最大目标值 =====
    model.objective = objective_rxn
    model.objective_direction = 'max'

    max_objective = model.slim_optimize()

    if max_objective is None or max_objective <= 1e-9:
        return model.optimize()

    # ===== 阶段2: 固定目标值，最小化酶池 =====
    target_flux = max_objective * fraction_of_optimum
    tolerance = abs(target_flux) * 0.001

    obj_rxn = model.reactions.get_by_id(objective_rxn)
    original_bounds = obj_rxn.bounds
    obj_rxn.bounds = (target_flux - tolerance, target_flux + tolerance)

    # 构建目标函数
    # 关键修复：对所有反应都加惩罚，没有kcat_MW的用小系数
    enzyme_cost_objective = {}
    default_penalty = 1e-9  # 对无kcat_MW反应的小惩罚

    for rxn in model.reactions:
        if hasattr(rxn, 'kcat_MW') and rxn.kcat_MW:
            try:
                kcat_mw = float(rxn.kcat_MW)
                if kcat_mw > 0:
                    # 有kcat_MW：使用酶成本
                    enzyme_cost_objective[rxn] = -1.0 / kcat_mw
                else:
                    # kcat_MW为0或负：使用小惩罚
                    enzyme_cost_objective[rxn] = -default_penalty
            except (ValueError, TypeError):
                enzyme_cost_objective[rxn] = -default_penalty
        else:
            # 无kcat_MW（如交换反应）：使用小惩罚，避免极端值
            enzyme_cost_objective[rxn] = -default_penalty

    model.objective = enzyme_cost_objective
    model.objective_direction = 'max'  # max(-cost) = min(cost)

    solution = model.optimize()

    # 恢复
    obj_rxn.bounds = original_bounds

    return solution


def compute_single_kcat(kcat_mw, model_dict, target_rxn, glucose_bounds,
                        reactions_to_monitor, atp_reactions):
    """使用最小化酶池策略计算单个kcat值"""
    import sys
    sys.path.insert(0, '.')

    from script.ECMpy_function import get_enzyme_constraint_model_with_reaction_set_exclusion
    import json
    import tempfile
    import os
    import numpy as np

    model_dict_copy = json.loads(json.dumps(model_dict))
    for rxn in model_dict_copy['reactions']:
        if rxn['id'] == target_rxn:
            rxn['kcat_MW'] = kcat_mw
            break

    tmp_path = None
    try:
        with tempfile.NamedTemporaryFile(suffix='.json', mode='w', delete=False) as tmp:
            json.dump(model_dict_copy, tmp, indent=2)
            tmp_path = tmp.name

        model = get_enzyme_constraint_model_with_reaction_set_exclusion(tmp_path)

        custom_constraints = {
            'rxn00006_c': (0,0),
            'EX_cpd00221_e': (0,5),
            'EX_cpd00027_e_reverse': glucose_bounds,
        }

        for rxn_id, bounds in custom_constraints.items():
            if rxn_id in model.reactions:
                model.reactions.get_by_id(rxn_id).bounds = bounds

        # 使用修复后的最小化酶池策略
        solution = optimize_minimize_enzyme_pool(model, 'EX_biomass_c', 1.0)

        if solution is not None and solution.status == 'optimal':
            row = {'kcat_MW': kcat_mw}

            for rxn_id in reactions_to_monitor + atp_reactions:
                row[rxn_id] = solution.fluxes.get(rxn_id, np.nan)

            flux_emp1 = row.get('rxn01100_c', 0)
            flux_emp2 = row.get('rxn00148_c_num1', 0)
            flux_synthase = row.get('rxn08173_c', 0)

            atp_info = calculate_atp_proportion(flux_emp1, flux_emp2, flux_synthase)

            if atp_info:
                row['ATP_EMP_percentage'], row['ATP_Synthase_percentage'], row['Total_ATP'] = atp_info
            else:
                row['ATP_EMP_percentage'] = row['ATP_Synthase_percentage'] = np.nan
                row['Total_ATP'] = 0

            return row
        return None

    except Exception as e:
        return None
    finally:
        if tmp_path and os.path.exists(tmp_path):
            try:
                os.remove(tmp_path)
            except:
                pass


def find_inactivation_boundary(df):
    """找到ATP合成酶反应通量为0的边界点"""
    if 'rxn08173_c' not in df.columns:
        return None

    threshold = 1e-6
    zero_flux_mask = df['rxn08173_c'].abs() < threshold

    if not zero_flux_mask.any():
        return None

    zero_indices = df[zero_flux_mask].index
    if len(zero_indices) > 0:
        return df.loc[zero_indices[0], 'kcat_MW']
    return None


def run_sensitivity_analysis(model_file, target_rxn='rxn08173_c',
                             kcat_range=(5000, 200), n_points=20,
                             glucose_bounds=(0, 5), n_jobs=None):
    """使用最小化酶池策略并行运行敏感性分析"""
    from joblib import Parallel, delayed

    if n_jobs is None:
        n_jobs = N_JOBS

    reactions_to_monitor = [
        'EX_biomass_c', 'rxn08287_c', 'rxn08173_c', 'rxnbccaa3_c', 'rxn08971_c',
        'EX_cpd00221_e', 'EX_cpd00007_e_reverse', 'EX_cpd00027_e_reverse'
    ]
    atp_reactions = ['rxn01100_c', 'rxn00148_c_num1', 'rxn08173_c']
    kcat_values = np.linspace(kcat_range[0], kcat_range[1], n_points)

    print("=" * 80)
    print(f"kcat_MW 敏感性分析 (最小化酶池策略-修复版)")
    print(f"优化目标: min Σ(1/kcat_MW × v) + ε×Σv")
    print(f"并行任务数: {n_jobs if n_jobs > 0 else '全部CPU'}")
    print(f"修改反应: {target_rxn}")
    print(f"kcat_MW 范围: {kcat_range[0]} → {kcat_range[1]}")
    print(f"葡萄糖约束: {glucose_bounds}")
    print("=" * 80)

    with open(model_file, 'r') as f:
        model_dict = json.load(f)

    results = Parallel(n_jobs=n_jobs, verbose=0, backend='loky')(
        delayed(compute_single_kcat)(
            kcat_mw, model_dict, target_rxn, glucose_bounds,
            reactions_to_monitor, atp_reactions
        )
        for kcat_mw in kcat_values
    )

    valid_results = [r for r in results if r is not None]

    df = pd.DataFrame(valid_results)
    if not df.empty:
        df = df.sort_values('kcat_MW', ascending=False).reset_index(drop=True)
    print(f"\n完成！有效结果: {len(valid_results)}/{n_points}")
    return df


def plot_combined_figure(df, save_path='figure_14067_minpool.png'):
    """绘制组合图（简化版：使用同一数据集）"""
    set_academic_style()
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

    inactivation_boundary = find_inactivation_boundary(df) or 300
    color_reduced = '#E3F2FD'
    color_inactive = '#FFEBEE'

    x_min, x_max = df['kcat_MW'].min(), df['kcat_MW'].max()

    # 子图A
    ax1 = axes[0]
    ax1.axvspan(x_max, inactivation_boundary, alpha=0.3, color=color_reduced, zorder=0)
    ax1.axvspan(inactivation_boundary, x_min, alpha=0.3, color=color_inactive, zorder=0)

    for rxn_id, label, color, marker in PLOT_CONFIG_GROUP1:
        if rxn_id in df.columns:
            ax1.plot(df['kcat_MW'], df[rxn_id], color=color, marker=marker,
                    label=label, linewidth=2.0, markersize=6, markeredgewidth=0, alpha=0.9, zorder=2)

    setup_clean_axes(ax1)
    ax1.set_xlabel(r'$k_{cat}$_MW', fontsize=11)
    ax1.set_ylabel(r'Reaction flux (mmol gDCW$^{-1}$ h$^{-1}$)', fontsize=11)
    ax1.invert_xaxis()
    y_max = df[['EX_biomass_c', 'EX_cpd00221_e', 'EX_cpd00007_e_reverse', 'EX_cpd00027_e_reverse']].max().max()
    ax1.set_ylim(bottom=-0.5, top=y_max * 1.25)
    ax1.legend(loc='upper left', frameon=False, fontsize=8, ncol=2)

    y1_bottom, y1_top = ax1.get_ylim()
    text_y1 = y1_bottom + (y1_top - y1_bottom) * 0.08
    ax1.text((x_max + inactivation_boundary) / 2, text_y1, 'Reduced activity',
             ha='center', va='center', fontsize=8, color='#1565C0', style='italic')
    ax1.text((inactivation_boundary + x_min) / 2, text_y1, 'Inactivation',
             ha='center', va='center', fontsize=8, color='#C62828', style='italic')

    # 子图B
    ax2 = axes[1]
    ax2.axvspan(x_max, inactivation_boundary, alpha=0.3, color=color_reduced, zorder=0)
    ax2.axvspan(inactivation_boundary, x_min, alpha=0.3, color=color_inactive, zorder=0)

    if not df.empty and 'ATP_EMP_percentage' in df.columns:
        valid_df = df[df['ATP_EMP_percentage'].notna()].copy()
        if not valid_df.empty:
            ax2.plot(valid_df['kcat_MW'], valid_df['ATP_EMP_percentage'],
                    color=COLORS['coral'], marker='o', linewidth=2.0, markersize=6,
                    markeredgewidth=0, label='Substrate-level Phosphorylation', alpha=0.9, zorder=2)
            ax2.plot(valid_df['kcat_MW'], valid_df['ATP_Synthase_percentage'],
                    color=COLORS['teal'], marker='s', linewidth=2.0, markersize=6,
                    markeredgewidth=0, label='Oxidative Phosphorylation', alpha=0.9, zorder=2)
            ax2.axhline(y=50, color=COLORS['gray'], linestyle='--', linewidth=1.0, alpha=0.5, zorder=1)

    setup_clean_axes(ax2)
    ax2.set_xlabel(r'$k_{cat}$_MW', fontsize=11)
    ax2.set_ylabel('ATP synthesis contribution (%)', fontsize=11)
    ax2.invert_xaxis()
    ax2.set_ylim(-2, 102)
    ax2.set_yticks(np.arange(0, 101, 20))
    ax2.legend(loc='upper left', frameon=False, fontsize=8)

    y2_bottom, y2_top = ax2.get_ylim()
    text_y2 = y2_bottom + (y2_top - y2_bottom) * 0.08
    ax2.text((x_max + inactivation_boundary) / 2, text_y2, 'Reduced activity',
             ha='center', va='center', fontsize=8, color='#1565C0', style='italic')
    ax2.text((inactivation_boundary + x_min) / 2, text_y2, 'Inactivation',
             ha='center', va='center', fontsize=8, color='#C62828', style='italic')

    # 子图C
    ax3 = axes[2]
    ax3.axvspan(x_max, inactivation_boundary, alpha=0.3, color=color_reduced, zorder=0)
    ax3.axvspan(inactivation_boundary, x_min, alpha=0.3, color=color_inactive, zorder=0)

    for rxn_id, label, color, marker in PLOT_CONFIG_GROUP2:
        if rxn_id in df.columns:
            ax3.plot(df['kcat_MW'], df[rxn_id], color=color, marker=marker,
                    label=label, linewidth=2.0, markersize=6, markeredgewidth=0, alpha=0.9, zorder=2)

    setup_clean_axes(ax3)
    ax3.set_xlabel(r'$k_{cat}$_MW', fontsize=11)
    ax3.set_ylabel(r'Reaction flux (mmol gDCW$^{-1}$ h$^{-1}$)', fontsize=11)
    ax3.invert_xaxis()
    y3_max = df[['rxn08287_c', 'rxn08971_c', 'rxnbccaa3_c']].max().max()
    ax3.set_ylim(bottom=-0.5, top=y3_max * 1.25)
    ax3.legend(loc='upper left', frameon=False, fontsize=8)

    y3_bottom, y3_top = ax3.get_ylim()
    text_y3 = y3_bottom + (y3_top - y3_bottom) * 0.08
    ax3.text((x_max + inactivation_boundary) / 2, text_y3, 'Reduced activity',
             ha='center', va='center', fontsize=8, color='#1565C0', style='italic')
    ax3.text((inactivation_boundary + x_min) / 2, text_y3, 'Inactivation',
             ha='center', va='center', fontsize=8, color='#C62828', style='italic')

    plt.tight_layout()
    plt.savefig(save_path, dpi=600, bbox_inches='tight')
    plt.close()
    print(f"图像已保存: {save_path}")


def main():
    MODEL_FILE = "model_Calibration/DLKcat_Calibration/ecCZ870_DLKcat_Calibration_modify.json"

    if not os.path.exists(MODEL_FILE):
        print(f"模型文件不存在: {MODEL_FILE}")
        return

    print("\n运行敏感性分析（葡萄糖摄入: 0-1000）")
    df = run_sensitivity_analysis(
        model_file=MODEL_FILE,
        target_rxn='rxn08173_c',
        kcat_range=(800, 10),
        n_points=20,
        glucose_bounds=(0, 1000),
        n_jobs=N_JOBS
    )

    if df.empty:
        print("未获得有效结果")
        return

    plot_combined_figure(df)

    print("\n" + "=" * 100)
    print("分析完成！")
    print("生成的文件:")
    print("  - figure_14067_minpool.png (组合图)")

    print("=" * 100)

if __name__ == "__main__":
    main()


运行敏感性分析（葡萄糖摄入: 0-1000）
kcat_MW 敏感性分析 (最小化酶池策略-修复版)
优化目标: min Σ(1/kcat_MW × v) + ε×Σv
并行任务数: 全部CPU
修改反应: rxn08173_c
kcat_MW 范围: 800 → 10
葡萄糖约束: (0, 1000)

完成！有效结果: 20/20
图像已保存: figure_14067_minpool.png

分析完成！
生成的文件:
  - figure_14067_minpool.png (组合图)


# ecFactory_light_iBridge

In [2]:
import sys

sys.path.append(r'./MEResource/')
sys.path.append(r'./script/')

from ecFbridge_minpool import IntegratedTargetIdentifier
from ECMpy_function import *

identifier = IntegratedTargetIdentifier(
    model_file="./model_Calibration/DLKcat_Calibration/ecCZ870_DLKcat_Calibration_modify.json",
    product_id="cpd00054",
    enable_ibridge=True,
    improvement_threshold=0.1,
    exclusion_threshold=-3,
    enable_distance_filter=True,
    output_dir='prediction_target_ecFbridge(test)',
)

final_targets = identifier.run_integrated_pipeline()

Loading metabolic model...
识别反应集合...
总共识别出 772 个需要约束的反应集合
添加反应集合互斥约束...
成功添加了 2820 个反应集合互斥约束
Using biomass reaction: EX_biomass_c
Closed reaction: EX_cpd00197_e_reverse

Initialization complete:
  iBridge analysis: Enabled
  Improvement threshold: 0.1%
  Exclusion threshold: -3%
  Distance filter: Enabled

Starting Integrated Target Identification Pipeline

Step 1: Running ecFactory Analysis
Read LP format model from file C:\Users\Asus\AppData\Local\Temp\tmp92bjqw4q.lp
Reading time = 0.02 seconds
: 4481 rows, 8917 columns, 35950 nonzeros
Loading essential genes with caching mechanism...
Loaded 264 essential genes from cache
Calculating optimal carbon source constraint bound...
Calculating carbon uptake for product maximization...
Optimal constraint bound: 4.868579
Closed reaction: EX_cpd00197_e_reverse

ecFactory Initialization Complete
Model: GEM_model
Product: EX_cpd00054_e
Biomass: EX_biomass_c
Strategy: include_ties
Top %: 40.0%
Essential genes: 264
Enzyme constraints: Yes
Separate

In [2]:
import sys
sys.path.append(r'./MEResource/')
sys.path.append(r'./script/')
from single_target_tester import test_single_target, batch_test_targets

result = test_single_target(
    model_file='./model_Calibration/DLKcat_Calibration/ecCZ870_DLKcat_Calibration_modify.json',
    product_id='cpd00051',
    gene_id='CEY17_RS10245',
    modification='both',        # 'OE', 'KD', 或 'both'
    method='enzyme_pool'      # 'enzyme_pool' 或 'pfba'
)

加载模型: ./model_Calibration/DLKcat_Calibration/ecCZ870_DLKcat_Calibration_modify.json
生物量反应: EX_biomass_c
产物反应: EX_cpd00051_e
优化方法: enzyme_pool
找到 2143 个反应具有kcat_MW值

测试基因 CEY17_RS10245 的双向效果

测试基因 CEY17_RS10245 的 OE 效果

基因: CEY17_RS10245
名称: pyk
关联反应数: 1

关联反应:
--------------------------------------------------------------------------------
  rxn00148_c_num1
    kcat_MW: 1893.0545
    反应式: cpd00008_c + cpd00061_c + cpd00067_c --> cpd00002_c + cpd000...

计算野生型基线
最大生长率: 0.507511
最大产物产量: 1.362974
约束生长率 (25%): 0.126878
基线产量: 1.023560
基线产率: 0.351625

应用 OE 修改...
修改因子: 5.0x

修改了 1 个反应的kcat_MW:
  rxn00148_c_num1: 1893.0545 -> 9465.2726

测试结果

基因: CEY17_RS10245
修改类型: OE (5.0x)

指标                   基线              修改后             变化             
-----------------------------------------------------------------
产量                   1.023560        1.026934        +0.3297%       
产率                   0.351625        0.350242        -0.3932%       

------------------------------------------------

# ecFbridge_combination

In [ ]:
import sys
import os
sys.path.append(r'./MEResource/')
sys.path.append(r'./script/')
from ecFbridge_combination import IntegratedTargetIdentifierWithOptimization

identifier = IntegratedTargetIdentifierWithOptimization(
    model_file='./model_Calibration/DLKcat_Calibration/ecCZ844_DLKcat_Calibration_modify.json',
    product_id='cpd00036',
    improvement_threshold=0.1,
    exclusion_threshold=-5,
    output_dir='prediction_target_ecFbridge_combination',
    eta_threshold=1,
    enable_distance_filter=True,
    enable_ibridge=True,
    enable_combination_optimization=True
)

final_targets = identifier.run_integrated_pipeline()